# ASR Benchmark — Automated Pipeline

This notebook runs automatically on Kaggle (scheduled weekly).

## Flow
1. Check HuggingFace **registry repo** for datasets to benchmark
2. Detect **new datasets** (not yet benchmarked)
3. Run benchmark on 2 models: `vinai/PhoWhisper-large` + `openai/whisper-large-v3-turbo`
4. Push results to HuggingFace **results repo**
5. Dashboard on HuggingFace Spaces auto-updates

## Setup (one-time)
1. **GPU**: Settings > Accelerator > **GPU T4 x2**
2. **Internet**: Settings > Internet > **On**
3. **Secrets**: Settings > Secrets:
   - `HF_TOKEN` — your HuggingFace write token
   - `HF_RESULTS_REPO` — e.g. `your-username/asr-benchmark-results`
   - `HF_REGISTRY_REPO` — e.g. `your-username/asr-benchmark-registry`
4. **Schedule**: File > Schedule > **Weekly**

In [ ]:
!pip install -q transformers datasets[audio] jiwer accelerate soundfile librosa huggingface_hub tqdm pyyaml

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Configuration

In [ ]:
import os

# --- Load secrets from Kaggle ---
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    HF_RESULTS_REPO = secrets.get_secret("HF_RESULTS_REPO")
    HF_REGISTRY_REPO = secrets.get_secret("HF_REGISTRY_REPO")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    HF_RESULTS_REPO = os.environ.get("HF_RESULTS_REPO", "")
    HF_REGISTRY_REPO = os.environ.get("HF_REGISTRY_REPO", "")

# Login to HuggingFace
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace")
else:
    print("WARNING: No HF_TOKEN. Add it in Kaggle Settings > Secrets.")

# --- Models ---
MODELS = [
    {"id": "vinai/PhoWhisper-large", "language": "vietnamese", "category": "vietnamese-specific", "size": "1550M"},
    {"id": "openai/whisper-large-v3-turbo", "language": "vietnamese", "category": "multilingual", "size": "809M"},
]

# --- Fallback datasets (used if registry repo is empty/unavailable) ---
FALLBACK_DATASETS = [
    {"id": "thanhnew2001/VietSuperSpeech", "split": "validation"},
]

# --- Settings ---
BATCH_SIZE = 8
MAX_SAMPLES = None  # None = all samples. Set to e.g. 10 for quick testing.
GPU_PRICING = {"T4": 0.35, "P100": 0.46, "free_tier": 0.00}

print(f"Models: {[m['id'] for m in MODELS]}")
print(f"Results repo: {HF_RESULTS_REPO}")
print(f"Registry repo: {HF_REGISTRY_REPO}")

## Detect New Datasets

Checks the **registry repo** on HuggingFace for datasets to benchmark.
Compares against already-benchmarked datasets in the **results repo**.
Only runs benchmark on **new** datasets.

In [ ]:
import json
from huggingface_hub import hf_hub_download, HfApi

def get_registered_datasets(registry_repo):
    """Fetch datasets list from registry repo."""
    try:
        path = hf_hub_download(repo_id=registry_repo, filename="datasets.json", repo_type="dataset")
        with open(path) as f:
            return json.load(f)
    except Exception as e:
        print(f"Could not load registry: {e}")
        return []

def get_benchmarked_datasets(results_repo):
    """Fetch which datasets have already been benchmarked."""
    try:
        path = hf_hub_download(repo_id=results_repo, filename="benchmarked.json", repo_type="dataset")
        with open(path) as f:
            return json.load(f)
    except Exception:
        return {}

def mark_benchmarked(results_repo, dataset_id, split, timestamp):
    """Mark a dataset as benchmarked."""
    benchmarked = get_benchmarked_datasets(results_repo)
    benchmarked[f"{dataset_id}:{split}"] = timestamp
    api = HfApi()
    api.upload_file(
        path_or_fileobj=json.dumps(benchmarked, indent=2).encode(),
        path_in_repo="benchmarked.json",
        repo_id=results_repo, repo_type="dataset",
        commit_message=f"Mark {dataset_id}:{split} as benchmarked")

# --- Detect new datasets ---
registered = get_registered_datasets(HF_REGISTRY_REPO) if HF_REGISTRY_REPO else []
if not registered:
    print("No registry found. Using fallback datasets.")
    registered = FALLBACK_DATASETS

benchmarked = get_benchmarked_datasets(HF_RESULTS_REPO) if HF_RESULTS_REPO else {}

datasets_to_run = []
for ds in registered:
    key = f"{ds['id']}:{ds.get('split', 'validation')}"
    if key not in benchmarked:
        datasets_to_run.append(ds)
        print(f"  NEW: {key}")
    else:
        print(f"  Already benchmarked: {key} ({benchmarked[key]})")

if not datasets_to_run:
    print("\nNo new datasets to benchmark. Notebook will exit early.")
else:
    print(f"\n{len(datasets_to_run)} new dataset(s) to benchmark.")

## Benchmark Engine

In [ ]:
import gc
import io
import time
from datetime import datetime, timezone

import librosa
import numpy as np
import requests
from datasets import load_dataset
from huggingface_hub import hf_hub_url
from jiwer import wer, cer
from tqdm.auto import tqdm
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

TARGET_SR = 16000

def normalize_text(text):
    return " ".join(text.strip().lower().split())

def download_audio(audio_path, dataset_id):
    url = hf_hub_url(repo_id=dataset_id, filename=audio_path, repo_type="dataset")
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    arr, _ = librosa.load(io.BytesIO(resp.content), sr=TARGET_SR, mono=True)
    return arr

def load_asr_pipeline(model_id, language="vietnamese"):
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    print(f"Loading {model_id} on {device}...")
    model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, torch_dtype=dtype, low_cpu_mem_usage=True, use_safetensors=True)
    model.to(device)
    processor = AutoProcessor.from_pretrained(model_id)
    pipe = pipeline("automatic-speech-recognition", model=model, tokenizer=processor.tokenizer,
                    feature_extractor=processor.feature_extractor, torch_dtype=dtype, device=device)
    return pipe, model

def unload_model(pipe, model):
    del pipe, model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def benchmark_model(model_cfg, dataset, dataset_id, batch_size, max_samples):
    model_id = model_cfg["id"]
    language = model_cfg.get("language", "vietnamese")
    print(f"\n{'='*60}\n  {model_id}\n{'='*60}")
    
    load_start = time.time()
    pipe, model = load_asr_pipeline(model_id, language)
    model_load_time = time.time() - load_start
    
    num = min(len(dataset), max_samples) if max_samples else len(dataset)
    refs, preds, samples = [], [], []
    total_audio, total_infer = 0.0, 0.0
    
    for i in tqdm(range(0, num, batch_size), desc=model_id.split('/')[-1]):
        batch = dataset.select(list(range(i, min(i + batch_size, num))))
        arrs, texts, durs = [], [], []
        for s in batch:
            try:
                arrs.append(download_audio(s["audio"], dataset_id))
                texts.append(s["text"])
                durs.append(s.get("duration", 0.0))
            except Exception as e:
                print(f"Skip: {e}")
        if not arrs: continue
        
        t0 = time.time()
        results = pipe(arrs, batch_size=len(arrs), generate_kwargs={"language": language, "task": "transcribe"})
        infer_t = time.time() - t0
        total_infer += infer_t
        
        for j, (r, ref, dur) in enumerate(zip(results, texts, durs)):
            rn, pn = normalize_text(ref), normalize_text(r["text"])
            sw = wer(rn, pn) if rn else 0.0
            sc = cer(rn, pn) if rn else 0.0
            total_audio += dur
            refs.append(rn); preds.append(pn)
            samples.append({"index": i+j, "reference": rn, "prediction": pn,
                           "wer": round(sw, 4), "cer": round(sc, 4), "duration_s": round(dur, 2)})
    
    ow = wer(refs, preds) if refs else 1.0
    oc = cer(refs, preds) if refs else 1.0
    rtf = total_infer / total_audio if total_audio > 0 else None
    costs = {gpu: round((total_infer/3600)*price, 6) for gpu, price in GPU_PRICING.items()}
    
    print(f"  WER: {ow:.4f} | CER: {oc:.4f} | RTF: {rtf:.4f if rtf else 'N/A'} | Cost(T4): ${costs.get('T4',0):.6f}")
    unload_model(pipe, model)
    
    return {
        "model_id": model_id, "category": model_cfg.get("category", ""),
        "size_params": model_cfg.get("size", ""), "dataset_id": dataset_id,
        "metrics": {"wer": round(ow,4), "cer": round(oc,4), "num_samples": len(refs),
                    "total_audio_duration_s": round(total_audio,2), "total_inference_time_s": round(total_infer,2),
                    "model_load_time_s": round(model_load_time,2),
                    "real_time_factor": round(rtf,4) if rtf else None,
                    "avg_time_per_sample_s": round(total_infer/len(refs),4) if refs else None},
        "cost_estimate_usd": costs, "per_sample_results": samples,
    }

print("Benchmark engine ready.")

## Run Benchmark (only on new datasets)

In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
device = "cuda" if torch.cuda.is_available() else "cpu"
all_results = []
leaderboard = []

if not datasets_to_run:
    print("No new datasets. Skipping benchmark.")
else:
    for ds_cfg in datasets_to_run:
        dataset_id = ds_cfg["id"]
        split = ds_cfg.get("split", "validation")
        print(f"\nLoading dataset: {dataset_id} ({split})...")
        dataset = load_dataset(dataset_id, split=split)
        print(f"Loaded: {len(dataset)} samples")
        
        for model_cfg in MODELS:
            try:
                result = benchmark_model(model_cfg, dataset, dataset_id, BATCH_SIZE, MAX_SAMPLES)
                all_results.append(result)
                leaderboard.append({
                    "model_id": result["model_id"], "category": result["category"],
                    "size_params": result["size_params"], "dataset_id": result["dataset_id"],
                    "wer": result["metrics"]["wer"], "cer": result["metrics"]["cer"],
                    "rtf": result["metrics"]["real_time_factor"],
                    "inference_time_s": result["metrics"]["total_inference_time_s"],
                    "avg_time_per_sample_s": result["metrics"]["avg_time_per_sample_s"],
                    "cost_T4_usd": result["cost_estimate_usd"].get("T4", 0),
                    "cost_free_tier": result["cost_estimate_usd"].get("free_tier", 0),
                })
            except Exception as e:
                print(f"FAILED: {model_cfg['id']} - {e}")
                all_results.append({"model_id": model_cfg["id"], "dataset_id": dataset_id, "error": str(e)})
    
    leaderboard.sort(key=lambda x: x.get("wer", 999))
    print(f"\nBenchmark complete! {len(leaderboard)} entries in leaderboard.")

## Results & Leaderboard

In [ ]:
import pandas as pd

if leaderboard:
    lb_df = pd.DataFrame(leaderboard)
    lb_df.index = range(1, len(lb_df) + 1)
    lb_df.index.name = "Rank"
    print("\nLEADERBOARD")
    display(lb_df[["model_id", "category", "dataset_id", "wer", "cer", "rtf", "cost_T4_usd"]])
else:
    print("No new results to display.")

## Save & Push Results to HuggingFace

In [ ]:
if not leaderboard:
    print("No new results to push.")
else:
    # Save locally
    full_output = {
        "metadata": {
            "timestamp": timestamp, "device": device,
            "torch_version": torch.__version__,
            "num_models": len(MODELS), "num_datasets": len(datasets_to_run),
            "max_samples": MAX_SAMPLES, "batch_size": BATCH_SIZE,
        },
        "leaderboard": leaderboard,
        "detailed_results": all_results,
    }
    
    output_file = f"/kaggle/working/benchmark_{timestamp}.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(full_output, f, ensure_ascii=False, indent=2)
    print(f"Results saved: {output_file}")
    
    # Push to HuggingFace
    if HF_RESULTS_REPO and HF_TOKEN:
        from datasets import Dataset, DatasetDict
        from huggingface_hub import create_repo
        
        create_repo(repo_id=HF_RESULTS_REPO, repo_type="dataset", exist_ok=True)
        
        # --- Merge with existing results ---
        try:
            existing_lb = load_dataset(HF_RESULTS_REPO, split="leaderboard")
            existing_lb_list = existing_lb.to_list()
        except Exception:
            existing_lb_list = []
        
        try:
            existing_samples = load_dataset(HF_RESULTS_REPO, split="per_sample")
            existing_samples_list = existing_samples.to_list()
        except Exception:
            existing_samples_list = []
        
        # Merge leaderboard (append new entries)
        merged_lb = existing_lb_list + leaderboard
        
        # Merge per-sample
        new_samples = []
        for d in all_results:
            if "error" in d: continue
            for s in d.get("per_sample_results", []):
                new_samples.append({"model_id": d["model_id"], "dataset_id": d["dataset_id"], **s})
        merged_samples = existing_samples_list + new_samples
        
        # Push merged data
        lb_keys = ["model_id", "category", "size_params", "dataset_id",
                   "wer", "cer", "rtf", "inference_time_s",
                   "avg_time_per_sample_s", "cost_T4_usd", "cost_free_tier"]
        lb_ds = Dataset.from_dict({k: [e.get(k) for e in merged_lb] for k in lb_keys})
        
        sample_keys = ["model_id", "dataset_id", "index", "reference", "prediction", "wer", "cer", "duration_s"]
        if merged_samples:
            samples_ds = Dataset.from_dict({k: [s.get(k) for s in merged_samples] for k in sample_keys})
        else:
            samples_ds = Dataset.from_dict({k: [] for k in sample_keys})
        
        DatasetDict({"leaderboard": lb_ds, "per_sample": samples_ds}).push_to_hub(HF_RESULTS_REPO)
        
        # Upload raw JSON
        api = HfApi()
        api.upload_file(path_or_fileobj=output_file, path_in_repo="latest_benchmark.json",
                        repo_id=HF_RESULTS_REPO, repo_type="dataset", commit_message=f"Benchmark {timestamp}")
        
        # Mark datasets as benchmarked
        for ds_cfg in datasets_to_run:
            mark_benchmarked(HF_RESULTS_REPO, ds_cfg["id"], ds_cfg.get("split", "validation"), timestamp)
        
        # Update README
        lb_table = "| Rank | Model | Category | Dataset | WER | CER |\n"
        lb_table += "|------|-------|----------|---------|-----|-----|\n"
        sorted_lb = sorted(merged_lb, key=lambda x: x.get('wer', 999))
        for i, e in enumerate(sorted_lb, 1):
            lb_table += f"| {i} | `{e['model_id']}` | {e.get('category','')} | `{e['dataset_id']}` | {e['wer']:.4f} | {e['cer']:.4f} |\n"
        
        readme = f"""---\nlanguage: [vi]\ntags: [asr, benchmark, whisper, vietnamese]\npretty_name: ASR Benchmark Results\n---\n\n# ASR Benchmark Results\n\nLast updated: {timestamp}\n\n## Leaderboard\n\n{lb_table}\n\n## Usage\n\n```python\nfrom datasets import load_dataset\nlb = load_dataset(\"{HF_RESULTS_REPO}\", split=\"leaderboard\")\nprint(lb.to_pandas().sort_values(\"wer\"))\n```\n"""
        api.upload_file(path_or_fileobj=readme.encode(), path_in_repo="README.md",
                        repo_id=HF_RESULTS_REPO, repo_type="dataset", commit_message="Update README")
        
        print(f"\nResults pushed to https://huggingface.co/datasets/{HF_RESULTS_REPO}")
    else:
        print("Set HF_RESULTS_REPO and HF_TOKEN to auto-push results.")

## Visualizations

In [ ]:
import matplotlib.pyplot as plt

if leaderboard:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    colors = ['#2ecc71' if c == 'vietnamese-specific' else '#3498db' for c in lb_df['category']]
    
    axes[0].barh(lb_df['model_id'], lb_df['wer'], color=colors)
    axes[0].set_xlabel('WER (lower = better)'); axes[0].set_title('Word Error Rate')
    axes[0].invert_yaxis()
    
    axes[1].barh(lb_df['model_id'], lb_df['cer'], color=colors)
    axes[1].set_xlabel('CER (lower = better)'); axes[1].set_title('Character Error Rate')
    axes[1].invert_yaxis()
    
    rtf_df = lb_df.dropna(subset=['rtf'])
    rtf_colors = ['#2ecc71' if c == 'vietnamese-specific' else '#3498db' for c in rtf_df['category']]
    axes[2].barh(rtf_df['model_id'], rtf_df['rtf'], color=rtf_colors)
    axes[2].set_xlabel('RTF (lower = faster)'); axes[2].set_title('Real-Time Factor')
    axes[2].axvline(x=1.0, color='red', linestyle='--', label='Real-time')
    axes[2].legend(); axes[2].invert_yaxis()
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/benchmark_charts.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No new results to visualize.")

## Summary

In [ ]:
print("="*60)
print("BENCHMARK RUN SUMMARY")
print("="*60)
print(f"Timestamp: {timestamp}")
print(f"New datasets benchmarked: {len(datasets_to_run)}")
print(f"Models: {[m['id'] for m in MODELS]}")
if leaderboard:
    best = leaderboard[0]
    print(f"Best WER: {best['model_id']} ({best['wer']:.4f})")
print(f"Results repo: https://huggingface.co/datasets/{HF_RESULTS_REPO}")
print("\nTo add new datasets: edit datasets.json in your registry repo.")
print("Next scheduled run will auto-detect and benchmark them.")
print("="*60)